# LZW coding, part A: the core (string table, encoder, decoder)

**INF605 Digital Image Processing, Group 8.** Paper: T. A. Welch, "A technique for high-performance data compression," *Computer*, 1984. Also used: R. C. Gonzalez and R. E. Woods, *Digital Image Processing*, 4th ed., Section 8.5 (LZW coding), Examples 8.6 and 8.7.

This notebook is the core of our LZW work. It contains the encoder, the decoder, and tests that show both work exactly (what goes in comes back out, bit for bit). The other LZW parts build on it:

- **Part B** turns the list of code numbers into real bytes (bit packing) and counts the true file size.
- **Part C** measures compression against the entropy and PNG, and runs the parameter experiments.

**What this notebook does**

1. The idea in short
2. The encoder
3. Reproducing Welch's own example (his Figure 5 and Table 3)
4. The decoder, and the special case Welch warns about
5. Tests (including the book's Example 8.7)
6. Round trip on our test images
7. The two parameters: code width and what to do when the table is full
8. The interface for parts B and C

In [1]:
import os, time
import numpy as np
import cv2
import pandas as pd

## 1. The idea in short

LZW learns a **string table** while it reads the data. The table starts with every single symbol (for bytes, 256 entries, code numbers 0 to 255). The encoder keeps the longest string `w` that is already in the table and reads the next symbol `K`:

- if `w + K` is in the table, then `w` becomes `w + K` (keep extending);
- otherwise, **output the code of `w`**, add `w + K` to the table under the next free number, and start a new string with `K`.

Nothing is sent about the table. The decoder rebuilds the identical table from the codes alone.

**Prefix property** (Welch): for every string in the table, the same string without its last symbol is also in the table. So an entry can be stored as a pair (code of the prefix string, one extra symbol). This is how we store it: a table entry is `(prefix code, symbol) -> new code`. Python's `dict` is a hash table, so looking up a pair is fast. This is the lookup that Welch implements with hashing in his hardware.

## 2. The encoder

The function follows Welch's compression algorithm. Two parameters matter later:

- `max_bits`: codes are at most this many bits wide, so the table holds at most $2^{max\_bits}$ entries (Welch usually uses 12, so 4096).
- `when_full`: what to do once the table is full. `"freeze"` keeps the table as it is (the baseline). `"reset"` sends a special CLEAR code and starts the table again from scratch. In `"reset"` mode the code number `alphabet_size` is reserved for CLEAR, so new entries start one number later.

In [2]:
def lzw_encode(data, max_bits=12, alphabet_size=256, when_full="freeze"):
    # Encode a sequence of integers (0 .. alphabet_size-1, for example bytes) into a list of LZW codes.
    assert when_full in ("freeze", "reset")
    max_size = 1 << max_bits
    reset = when_full == "reset"
    clear = alphabet_size                              # only used in "reset" mode
    first_new = alphabet_size + (1 if reset else 0)    # first code number given to a learned string

    table = {}              # (prefix code, symbol) -> code of the longer string
    next_code = first_new
    out = []
    it = iter(data)
    try:
        w = next(it)        # w: code of the current string (it starts as a single symbol, whose code is the symbol)
    except StopIteration:
        return out          # empty input
    for K in it:
        key = (w, K)
        if key in table:                # w + K is known: keep extending
            w = table[key]
        else:
            out.append(w)               # output the code of w
            if next_code < max_size:
                table[key] = next_code  # learn w + K
                next_code += 1
            elif reset:
                out.append(clear)       # table full: tell the decoder to start over
                table.clear()
                next_code = first_new
            w = K                       # start a new string with K
    out.append(w)                       # input ended: output the last string
    return out

## 3. Reproducing Welch's own example

Welch's Figure 5 compresses the 17-symbol input `a b a b c b a b a b a a a a a a a` with the alphabet {a, b, c} (codes 1, 2, 3, new strings from 4 on) and Table 3 lists the strings that get learned. The function below runs the same steps on text strings, so we can print every step and compare with the paper. Codes are numbered from 1 here, as in the paper. (Our encoder numbers from 0, so its codes are one smaller.)

In [3]:
def lzw_trace(text, alphabet):
    # Same algorithm on strings, numbered from 1 like the paper. Returns the step table, the codes, the string table.
    table = {ch: i + 1 for i, ch in enumerate(alphabet)}
    rows, out = [], []
    w = text[0]
    for K in text[1:]:
        if w + K in table:
            rows.append((w, K, "yes", "", ""))
            w = w + K
        else:
            out.append(table[w])
            table[w + K] = len(table) + 1
            rows.append((w, K, "no", table[w], f"{w + K} = {len(table)}"))
            w = K
    out.append(table[w])
    rows.append((w, "(end)", "", table[w], ""))
    return pd.DataFrame(rows, columns=["w", "K", "w+K in table?", "output code", "new table entry"]), out, table

text = "ababcbababaaaaaaa"
steps, paper_codes, paper_table = lzw_trace(text, "abc")
steps

,w,K,w+K in table?,output code,new table entry
0,a,b,no,1,ab = 4
1,b,a,no,2,ba = 5
2,a,b,yes,,
3,ab,c,no,4,abc = 6
4,c,b,no,3,cb = 7
5,b,a,yes,,
6,ba,b,no,5,bab = 8
7,b,a,yes,,
8,ba,b,yes,,
9,bab,a,no,8,baba = 9


In [4]:
print("output codes:", paper_codes)
print("learned strings:", {s: c for s, c in paper_table.items() if len(s) > 1})

# Welch's Figure 5 shows the codes 1 2 4 3 5 8 1 10 11 (the last code runs off the edge of the page; the algorithm gives a final 1).
assert paper_codes == [1, 2, 4, 3, 5, 8, 1, 10, 11, 1]
# Welch's Table 3: ab=4 ba=5 abc=6 cb=7 bab=8 baba=9 aa=10 aaa=11 aaaa=12  (the scan prints "aa" for entry 11; it must be "aaa")
assert {s: c for s, c in paper_table.items() if len(s) > 1} == \
       {"ab": 4, "ba": 5, "abc": 6, "cb": 7, "bab": 8, "baba": 9, "aa": 10, "aaa": 11, "aaaa": 12}

# the real encoder (numbers from 0, symbols a,b,c = 0,1,2) gives the same codes, shifted by one
symbols = [ "abc".index(ch) for ch in text ]
assert [c + 1 for c in lzw_encode(symbols, alphabet_size=3)] == paper_codes
print("matches Welch's Figure 5 and Table 3")

output codes: [1, 2, 4, 3, 5, 8, 1, 10, 11, 1]
learned strings: {'ab': 4, 'ba': 5, 'abc': 6, 'cb': 7, 'bab': 8, 'baba': 9, 'aa': 10, 'aaa': 11, 'aaaa': 12}
matches Welch's Figure 5 and Table 3


## 4. The decoder, and the special case

The decoder reads a code, looks up the string, outputs it, and adds **one new entry**: the *previous* string plus the *first symbol of the current string*. This rebuilds exactly the entries the encoder created, one step later. A string is looked up by walking back through the prefixes (an entry is "prefix code + one symbol"), which is what Welch's Figure 6 shows: code 5 becomes code 2 plus `a`, and code 2 becomes `b`.

**The special case** (Welch calls it KwKwK): the encoder can use an entry in the very step after creating it, before the decoder has had the chance to add it. Then the code is **not yet in the decoder's table** (it equals the next free number). It is still decodable: the unknown string is the previous string plus its own first symbol. In Welch's example this happens when the decoder receives code 8: the decoder has only just added code 7, so it knows `ba` (code 5) and builds `ba` + `b` = `bab`.

In [5]:
def lzw_decode(codes, max_bits=12, alphabet_size=256, when_full="freeze"):
    # Rebuild the data from the codes. Must use the same parameters as the encoder. Returns a list of integers.
    assert when_full in ("freeze", "reset")
    max_size = 1 << max_bits
    reset = when_full == "reset"
    clear = alphabet_size
    first_new = alphabet_size + (1 if reset else 0)

    prefix = [None] * max_size          # entry = (prefix code, one extra symbol)
    suffix = [None] * max_size
    next_code = first_new

    def string_of(code):
        # walk back through the prefixes (Welch's decoder uses a stack for this), then reverse
        s = []
        while code >= first_new:
            s.append(suffix[code])
            code = prefix[code]
        s.append(code)                  # a single symbol: its code is the symbol itself
        s.reverse()
        return s

    out = []
    prev = None                         # code read in the previous step
    for code in codes:
        if reset and code == clear:     # CLEAR: forget the table, the next code is a single symbol again
            next_code, prev = first_new, None
            continue
        if prev is None:
            if code >= alphabet_size:
                raise ValueError("stream must start with a single-symbol code")
            s = [code]
        elif code < next_code:
            s = string_of(code)
        elif code == next_code and next_code < max_size:
            s = string_of(prev)
            s = s + s[:1]               # the special case: previous string + its own first symbol
        else:
            raise ValueError(f"invalid code {code}")
        out.extend(s)
        if prev is not None and next_code < max_size:
            prefix[next_code], suffix[next_code] = prev, s[0]   # previous string + first symbol of this one
            next_code += 1
        prev = code
    return out

## 5. Tests

First the small cases (including Welch's example, the special case, and the book's Example 8.7: a 4 x 4 image whose 16 pixels become ten 9-bit codes, 90 bits instead of 128, ratio 1.42), then a battery of round trips on edge cases and several parameter settings. The rule is always `decode(encode(x)) == x`.

In [6]:
# Welch's example through the real functions
codes = lzw_encode(symbols, alphabet_size=3)
assert lzw_decode(codes, alphabet_size=3) == symbols
print("Welch's example round trip: OK")

# small hand-checked cases with bytes
assert lzw_encode(b"ABABABA") == [65, 66, 256, 258]
assert lzw_encode(b"AAAAAAA") == [65, 256, 257, 65]    # the decoder meets the special case twice (codes 256 and 257)
assert bytes(lzw_decode([65, 256, 257, 65])) == b"AAAAAAA"
# the book's Example 8.7 (Gonzalez & Woods, Section 8.5): 4x4 image, 9-bit codes, pixels read left to right, top to bottom
edge = np.array([[39, 39, 126, 126]] * 4, dtype=np.uint8)
edge_codes = lzw_encode(edge.tobytes(), max_bits=9)
assert edge_codes == [39, 39, 126, 126, 256, 258, 260, 259, 257, 126]    # the book's "Encoded Output" column
assert len(edge_codes) * 9 == 90 and round(128 / 90, 2) == 1.42          # 90 bits instead of 128, ratio 1.42:1
assert bytes(lzw_decode(edge_codes, max_bits=9)) == edge.tobytes()
print("hand-checked cases and the book's Example 8.7: OK")

# round-trip battery
rng = np.random.default_rng(1)
samples = {
    "empty": b"",
    "one byte": b"A",
    "all 256 values": bytes(range(256)),
    "10000 zero bytes": bytes(10000),
    "text": ("to be or not to be, that is the question. " * 200).encode(),
    "random bytes": rng.integers(0, 256, 20000, dtype=np.uint8).tobytes(),
}
checked = 0
for name, data in samples.items():
    for max_bits in (9, 10, 12, 16):
        for policy in ("freeze", "reset"):
            codes = lzw_encode(data, max_bits, when_full=policy)
            assert all(c < (1 << max_bits) for c in codes), (name, max_bits, policy)
            assert bytes(lzw_decode(codes, max_bits, when_full=policy)) == data, (name, max_bits, policy)
            checked += 1
print(f"{checked} round trips (6 inputs x 4 code widths x 2 policies): OK")

Welch's example round trip: OK
hand-checked cases and the book's Example 8.7: OK


48 round trips (6 inputs x 4 code widths x 2 policies): OK


## 6. Round trip on our test images

An image is turned into bytes row by row (`img.tobytes()`, left to right, top to bottom) and decoded back to the same array. The "ratio (naive)" column assumes every code takes `max_bits` bits, so it ignores headers and bit packing. The real file size is part B's job.

In [7]:
IMAGE_DIR = next(p for p in ("../images", "images") if os.path.isdir(p))

def load_gray(path):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img

rows = []
for fname in sorted(os.listdir(IMAGE_DIR)):
    if not fname.lower().endswith((".tif", ".tiff")):
        continue
    img = load_gray(os.path.join(IMAGE_DIR, fname))
    data = img.tobytes()
    t = time.time()
    codes = lzw_encode(data, max_bits=12)
    back = np.frombuffer(bytes(lzw_decode(codes, max_bits=12)), dtype=np.uint8).reshape(img.shape)
    assert np.array_equal(back, img), fname
    rows.append({"image": fname, "bytes": len(data), "codes": len(codes),
                 "bytes per code": round(len(data) / len(codes), 2),
                 "ratio (naive, 12-bit codes)": round(len(data) * 8 / (len(codes) * 12), 2),
                 "seconds": round(time.time() - t, 1)})
pd.DataFrame(rows).set_index("image")

,bytes,codes,bytes per code,"ratio (naive, 12-bit codes)",seconds
image,,,,,
2.1.11.tiff,262144,229480,1.14,0.76,0.3
4.2.01.tiff,262144,217473,1.21,0.80,0.2
4.2.03.tiff,262144,220853,1.19,0.79,0.2
5.1.09.tiff,65536,44682,1.47,0.98,0.1
5.3.01.tiff,1048576,748960,1.40,0.93,0.9
7.1.10.tiff,262144,134447,1.95,1.30,0.2
texmos1.p512.tiff,262144,233170,1.12,0.75,0.2
texmos3b.p512.tiff,262144,224654,1.17,0.78,0.2


Every image comes back exactly. A ratio below 1 means the 12-bit codes take **more** space than the 8-bit pixels. That happens on most of our images: repeated pixel strings are short, so each code (12 bits) stands for only a little more than one pixel (8 bits). The desert image has longer repeats (about 2 pixels per code) and gets above 1. Welch describes the same effect at the start of a message: with 12-bit codes and 8-bit symbols, early codes stand for one symbol each, which is a 50 percent expansion. For comparison, the book's Example 8.6 reports a ratio of 1.28 for TIFF's LZW option on its own 512 x 512 image. Parts B and C will look at this properly.

## 7. The two parameters

**`max_bits`** limits the table size. The book states the trade-off: if the dictionary is too small, matching sequences are less likely to be found, and if it is too large, the wider code words hurt compression. **`when_full`** decides what happens after the table is full. The book's simple solution is to flush (reinitialize) the dictionary when it is full, which is our `"reset"`. It also mentions watching the compression performance and flushing only when it gets poor, or replacing the least used entries. Our `"freeze"` keeps the table as it is. The cell below shows both parameters working on one image: with a 9-bit table (512 entries) the table is full almost immediately.

In [8]:
img = load_gray(os.path.join(IMAGE_DIR, "7.1.10.tiff"))
data = img.tobytes()
rows = []
for max_bits in (9, 12, 16):
    for policy in ("freeze", "reset"):
        codes = lzw_encode(data, max_bits, when_full=policy)
        assert bytes(lzw_decode(codes, max_bits, when_full=policy)) == data
        rows.append({"max_bits": max_bits, "when_full": policy, "codes": len(codes),
                     "ratio (naive)": round(len(data) * 8 / (len(codes) * max_bits), 3)})
pd.DataFrame(rows).set_index(["max_bits", "when_full"])

codes  ratio (naive)
max_bits when_full                       
9        freeze     211382          1.102
         reset      216308          1.077
12       freeze     134447          1.300
         reset      140509          1.244
16       freeze      92094          1.423
         reset       97025          1.351

## 8. The interface for parts B and C

```python
lzw_encode(data, max_bits=12, alphabet_size=256, when_full="freeze") -> list[int]
lzw_decode(codes, max_bits=12, alphabet_size=256, when_full="freeze") -> list[int]
```

- `data` is a sequence of integers in `range(alphabet_size)`, for example `img.tobytes()` for an 8-bit image.
- The output is a list of code numbers, each smaller than `2**max_bits`. In `"reset"` mode the number `alphabet_size` is the CLEAR code.
- The decoder needs the same `max_bits`, `alphabet_size` and `when_full` as the encoder, and these go into the file header in part B.
- Notes for **part B** (bit packing): if the code width grows (9 bits at first, then 10, ... up to `max_bits`), the width of the $k$-th code (counting from 1) is set by the largest value it can have. In `"freeze"` mode that is `alphabet_size + k - 2` for $k \ge 2$ (the decoder has added one entry fewer than the encoder), capped at `2**max_bits - 1`; the first code is a single symbol. Welch describes the same idea (with 8-bit characters the first 256 codes need only 9 bits, the next 512 codes 10 bits, and so on) and reports about a 7 percent better ratio than fixed-width codes for a ten-thousand-character message at two-to-one compression. The cell below checks the bound. In `"reset"` mode the count starts again after every CLEAR code.
- Notes for **part C**: Welch's Figure 5 and the book's Example 8.7 are already reproduced (Sections 3 and 5). Welch's Table 2 (ratios by data type) can only be compared in spirit, because it was measured on different data. His Figure 3 (effect of block size on compression efficiency) can be reproduced in spirit: compress independent blocks of different lengths, since the table starts empty for every block, and watch the ratio.

In [9]:
# the bound that part B can use to choose the code width
codes = lzw_encode(img.tobytes(), max_bits=12)
assert all(c <= min(256 + k - 2, 4095) for k, c in enumerate(codes, start=1) if k >= 2)
print("each code stays within its width bound:", len(codes), "codes")

each code stays within its width bound: 134447 codes
